In [1]:
import os
import json
import datetime
import random
import zipfile
import numpy as np
import pandas as pd
import torch
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    confusion_matrix,
    classification_report,
)
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import Dataset
from transformers import (
    AutoConfig,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    get_cosine_schedule_with_warmup,
)
from transformers import logging as hf_logging

hf_logging.set_verbosity_error()

import warnings

warnings.filterwarnings("ignore")

os.environ["PYTHONWARNINGS"] = "ignore"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seeds = [42, 1337, 2026]

data_dir = r"C:\Users\X\Desktop\X\ALTA\alta2026_public_data"
root_dir = r"C:\Users\X\Desktop\X\ALTA"
test_csv = r"C:\Users\X\Desktop\X\ALTA\sample_test\answer.csv"

model_name = "microsoft/deberta-v3-large"
max_length = 256
grad_accum_steps = 4

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

task_configs = {
    "sentiment": {
        "epochs": 8,
        "batch_size": 4,
        "lr": 1.0e-5,
        "patience": 3,
        "label_smoothing": 0.05,
        "layer_decay": 0.9,
        "classifier_dropout": 0.1,
        "weight_key": "variety",
        "extra_multiplier": None,
    },
    "sarcasm": {
        "epochs": 12,
        "batch_size": 4,
        "lr": 8.0e-6,
        "patience": 4,
        "label_smoothing": 0.1,
        "layer_decay": 0.85,
        "classifier_dropout": 0.2,
        "weight_key": "group_key",
        "extra_multiplier": None,
    },
}

run_id = (
    datetime.datetime.now().strftime("%Y%m%d-%H%M%S") + f"_{model_name.split('/')[-1]}_ensemble"
)

run_dir = os.path.join(root_dir, "runs", run_id)
work_dir = os.path.join(run_dir, "tf_runs")
model_out_dir = os.path.join(run_dir, "final_models")
metrics_dir = os.path.join(run_dir, "metrics")
out = os.path.join(run_dir, "answer.csv")
submission_zip = os.path.join(run_dir, "submission.zip")

for d in (run_dir, work_dir, model_out_dir, metrics_dir):
    os.makedirs(d, exist_ok=True)

run_config = {
    "run_id": run_id,
    "timestamp": datetime.datetime.now().isoformat(),
    "model_name": model_name,
    "max_length": max_length,
    "grad_accum_steps": grad_accum_steps,
    "seeds": seeds,
    "task_configs": task_configs,
    "data_dir": data_dir,
    "use_bf16": use_bf16,
    "use_fp16": use_fp16,
}

with open(os.path.join(run_dir, "run_config.json"), "w") as f:
    json.dump(run_config, f, indent=2)


class TextDataset(Dataset):
    def __init__(self, encodings, labels=None, sample_weights=None):
        self.encodings = encodings
        self.labels = labels
        self.sample_weights = sample_weights

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}

        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx])

        if self.sample_weights is not None:
            item["sample_weight"] = torch.tensor(self.sample_weights[idx], dtype=torch.float)

        return item


class WeightedCollator(DataCollatorWithPadding):
    def __call__(self, features):
        weights = [f.pop("sample_weight") for f in features if "sample_weight" in f]

        batch = super().__call__(features)

        if weights:
            batch["sample_weight"] = torch.stack(weights)

        return batch


def make_weighted_trainer(label_smoothing):
    class WeightedTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            labels = inputs.pop("labels")
            weight = inputs.pop("sample_weight", None)

            outputs = model(**inputs)
            logits = outputs.logits

            per_sample = torch.nn.functional.cross_entropy(
                logits, labels, reduction="none", label_smoothing=label_smoothing
            )

            if weight is not None:
                weight = weight.to(logits.device)
                loss = (per_sample * weight).sum() / weight.sum()
            else:
                loss = per_sample.mean()

            return (loss, outputs) if return_outputs else loss

    return WeightedTrainer


def get_device():
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"


def grouped_class_weights(labels, group, extra_multiplier=None):
    labels = np.array(labels)
    group = np.array(group)
    weights = np.ones(len(labels), dtype=np.float32)

    for g in np.unique(group):
        mask = group == g

        cw = compute_class_weight(class_weight="balanced", classes=np.array([0, 1]), y=labels[mask])

        for c in (0, 1):
            w = cw[c]

            if extra_multiplier and (g, c) in extra_multiplier:
                w *= extra_multiplier[(g, c)]

            weights[mask & (labels == c)] = w

    return weights


def make_compute_metrics(variety_array):
    variety_array = np.asarray(variety_array)
    varieties = sorted(set(variety_array))

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)

        result = {"macro_f1": f1_score(labels, preds, average="macro")}

        per_variety = []

        for v in varieties:
            mask = variety_array == v

            f1 = f1_score(labels[mask], preds[mask], average="macro")

            result[f"macro_f1_{v}"] = f1
            per_variety.append(f1)

        result["weak_macro_f1"] = min(per_variety)

        return result

    return compute_metrics


def build_optimizer(model, base_lr, layer_decay, weight_decay=0.01):
    no_decay = ["bias", "LayerNorm.weight", "layer_norm.weight"]

    num_layers = model.config.num_hidden_layers
    groups = {}

    def layer_id_of(name):
        if "embeddings" in name:
            return 0

        if ".layer." in name:
            return int(name.split(".layer.")[1].split(".")[0]) + 1

        return num_layers + 1

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue

        layer_id = layer_id_of(name)

        scale = layer_decay ** (num_layers + 1 - layer_id)

        decay = 0.0 if any(nd in name for nd in no_decay) else weight_decay

        key = (layer_id, decay)

        if key not in groups:
            groups[key] = {"params": [], "lr": base_lr * scale, "weight_decay": decay}

        groups[key]["params"].append(param)

    return torch.optim.AdamW(list(groups.values()), lr=base_lr)


def train_one_task(task, cfg, train_df, valid_df, test_df, tokenizer, device, seed):
    train_enc = tokenizer(list(train_df["text"]), truncation=True, max_length=max_length)

    valid_enc = tokenizer(list(valid_df["text"]), truncation=True, max_length=max_length)

    test_enc = tokenizer(list(test_df["text"]), truncation=True, max_length=max_length)

    train_labels = train_df[task].tolist()

    sample_weights = grouped_class_weights(
        train_labels, train_df[cfg["weight_key"]].tolist(), extra_multiplier=cfg["extra_multiplier"]
    )

    train_ds = TextDataset(train_enc, train_labels, sample_weights)

    valid_ds = TextDataset(valid_enc, valid_df[task].tolist())

    test_ds = TextDataset(test_enc)

    config = AutoConfig.from_pretrained(model_name, num_labels=2)

    config.classifier_dropout = cfg["classifier_dropout"]
    config.hidden_dropout_prob = cfg["classifier_dropout"]

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        config=config,
        torch_dtype=torch.float32,
    )

    model.to(device)

    model.gradient_checkpointing_enable()

    assert all(p.dtype == torch.float32 for p in model.parameters())

    collator = WeightedCollator(tokenizer)

    optimizer = build_optimizer(model, cfg["lr"], cfg["layer_decay"])

    effective_batch_size = cfg["batch_size"] * grad_accum_steps

    steps_per_epoch = max(1, int(np.ceil(len(train_ds) / effective_batch_size)))

    total_steps = steps_per_epoch * cfg["epochs"]
    warmup_steps = int(0.1 * total_steps)

    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    training_args = TrainingArguments(
        output_dir=os.path.join(work_dir, f"{task}_seed{seed}"),
        num_train_epochs=cfg["epochs"],
        per_device_train_batch_size=cfg["batch_size"],
        per_device_eval_batch_size=cfg["batch_size"] * 2,
        gradient_accumulation_steps=grad_accum_steps,
        learning_rate=cfg["lr"],
        warmup_ratio=0.1,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="weak_macro_f1",
        greater_is_better=True,
        logging_steps=25,
        fp16=use_fp16,
        bf16=use_bf16,
        gradient_checkpointing=True,
        eval_accumulation_steps=1,
        max_grad_norm=1.0,
        seed=seed,
        data_seed=seed,
        report_to=[],
        remove_unused_columns=False,
    )

    WeightedTrainer = make_weighted_trainer(cfg["label_smoothing"])

    trainer = WeightedTrainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=valid_ds,
        data_collator=collator,
        compute_metrics=make_compute_metrics(valid_df["variety"].to_numpy()),
        optimizers=(optimizer, scheduler),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=cfg["patience"])],
    )

    trainer.train()

    save_path = os.path.join(model_out_dir, f"{task}_seed{seed}")

    os.makedirs(save_path, exist_ok=True)

    trainer.save_model(save_path)
    tokenizer.save_pretrained(save_path)

    with open(os.path.join(metrics_dir, f"{task}_seed{seed}_trainer_log_history.json"), "w") as f:
        json.dump(trainer.state.log_history, f, indent=2)

    valid_preds = trainer.predict(valid_ds)
    test_preds = trainer.predict(test_ds)

    valid_probs = torch.softmax(torch.tensor(valid_preds.predictions), dim=-1).numpy()
    test_probs = torch.softmax(torch.tensor(test_preds.predictions), dim=-1).numpy()

    del model, trainer
    torch.cuda.empty_cache()

    return valid_probs, test_probs


def plot_confusion_matrix(cm, class_names, title, save_path):
    fig, ax = plt.subplots(figsize=(4, 4))

    im = ax.imshow(cm, cmap="Blues")

    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

    ax.set_xticks(range(len(class_names)))
    ax.set_yticks(range(len(class_names)))

    ax.set_xticklabels(class_names)
    ax.set_yticklabels(class_names)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(
                j,
                i,
                str(cm[i, j]),
                ha="center",
                va="center",
                color=("white" if cm[i, j] > cm.max() / 2 else "black"),
            )

    fig.colorbar(im, ax=ax)

    fig.tight_layout()

    fig.savefig(save_path, dpi=150)

    plt.close(fig)


def compute_all_metrics(gold_df, run_df):
    all_metrics = {}
    official_scores = {}
    class_names = ["0", "1"]

    for task in ["sentiment", "sarcasm"]:
        all_metrics[task] = {}

        for dialect in ["en-AU", "en-UK"]:
            gold = gold_df[gold_df["variety"] == dialect][task]

            pred = run_df[run_df["variety"] == dialect][task]

            macro_f1 = f1_score(gold, pred, average="macro")

            macro_precision = precision_score(gold, pred, average="macro", zero_division=0)

            macro_recall = recall_score(gold, pred, average="macro", zero_division=0)

            acc = accuracy_score(gold, pred)

            report = classification_report(gold, pred, output_dict=True, zero_division=0)

            cm = confusion_matrix(gold, pred, labels=[0, 1])

            all_metrics[task][dialect] = {
                "accuracy": acc,
                "macro_f1": macro_f1,
                "macro_precision": macro_precision,
                "macro_recall": macro_recall,
                "classification_report": report,
                "confusion_matrix": cm.tolist(),
            }

            official_scores[f"{task}-{dialect}"] = macro_f1

            cm_df = pd.DataFrame(
                cm,
                index=[f"true_{c}" for c in class_names],
                columns=[f"pred_{c}" for c in class_names],
            )

            cm_df.to_csv(os.path.join(metrics_dir, f"cm_{task}_{dialect}.csv"))

            plot_confusion_matrix(
                cm,
                class_names,
                title=f"{task} — {dialect}",
                save_path=os.path.join(metrics_dir, f"cm_{task}_{dialect}.png"),
            )

        gold_all = gold_df[task]
        pred_all = run_df[task]

        cm_all = confusion_matrix(gold_all, pred_all, labels=[0, 1])

        all_metrics[task]["overall"] = {
            "accuracy": accuracy_score(gold_all, pred_all),
            "macro_f1": f1_score(gold_all, pred_all, average="macro"),
            "macro_precision": precision_score(
                gold_all, pred_all, average="macro", zero_division=0
            ),
            "macro_recall": recall_score(gold_all, pred_all, average="macro", zero_division=0),
            "classification_report": classification_report(
                gold_all, pred_all, output_dict=True, zero_division=0
            ),
            "confusion_matrix": cm_all.tolist(),
        }

        cm_df_all = pd.DataFrame(
            cm_all,
            index=[f"true_{c}" for c in class_names],
            columns=[f"pred_{c}" for c in class_names],
        )

        cm_df_all.to_csv(os.path.join(metrics_dir, f"cm_{task}_overall.csv"))

        plot_confusion_matrix(
            cm_all,
            class_names,
            title=f"{task} — overall",
            save_path=os.path.join(metrics_dir, f"cm_{task}_overall.png"),
        )

    final = (
        min(official_scores["sentiment-en-AU"], official_scores["sentiment-en-UK"])
        + min(official_scores["sarcasm-en-AU"], official_scores["sarcasm-en-UK"])
    ) / 2

    all_metrics["official_weak_macro_f1_per_task_dialect"] = official_scores

    all_metrics["official_final_score"] = final

    return all_metrics, final


device = get_device()

print(f"Run ID: {run_id}")
print(f"Outputs will be written under: {run_dir}")
print(f"Using device: {device}")
print(f"Precision: bf16={use_bf16} " f"fp16={use_fp16}")
print(f"Model: {model_name}")
print(f"Gradient accumulation: {grad_accum_steps}")
print(f"Seeds: {seeds}")

train_df = pd.read_csv(os.path.join(data_dir, "train.csv"))

valid_df = pd.read_csv(os.path.join(data_dir, "valid.csv"))

test_df = pd.read_csv(test_csv)

for d in (train_df, valid_df, test_df):
    d["group_key"] = d["source"] + "_" + d["variety"]
    d["text"] = "[" + d["source"] + "/" + d["variety"] + "] " + d["text"]

tokenizer = AutoTokenizer.from_pretrained(model_name)

valid_probs_by_seed = {task: [] for task in ["sentiment", "sarcasm"]}
test_probs_by_seed = {task: [] for task in ["sentiment", "sarcasm"]}

for s in seeds:
    print(f"\nSeed {s} =")
    set_seed(s)

    for task in ["sentiment", "sarcasm"]:
        print(f"\nFinetuning for task: {task} (seed {s})")

        vp, tp = train_one_task(
            task, task_configs[task], train_df, valid_df, test_df, tokenizer, device, s
        )

        valid_probs_by_seed[task].append(vp)
        test_probs_by_seed[task].append(tp)

        np.save(os.path.join(metrics_dir, f"{task}_seed{s}_valid_probs.npy"), vp)
        np.save(os.path.join(metrics_dir, f"{task}_seed{s}_test_probs.npy"), tp)

valid_preds = {}
test_preds = {}

for task in ["sentiment", "sarcasm"]:
    valid_avg_probs = np.mean(np.stack(valid_probs_by_seed[task], axis=0), axis=0)
    test_avg_probs = np.mean(np.stack(test_probs_by_seed[task], axis=0), axis=0)

    valid_preds[task] = np.argmax(valid_avg_probs, axis=-1)
    test_preds[task] = np.argmax(test_avg_probs, axis=-1)

    np.save(os.path.join(metrics_dir, f"{task}_ensemble_valid_probs.npy"), valid_avg_probs)
    np.save(os.path.join(metrics_dir, f"{task}_ensemble_test_probs.npy"), test_avg_probs)

valid_answer_df = valid_df[["source", "variety", "text"]].copy()

valid_answer_df["sentiment"] = valid_preds["sentiment"]

valid_answer_df["sarcasm"] = valid_preds["sarcasm"]

test_answer_df = test_df[["source", "variety", "text"]].copy()

test_answer_df["sentiment"] = test_preds["sentiment"]

test_answer_df["sarcasm"] = test_preds["sarcasm"]

test_answer_df.to_csv(out, index=False)

print(f"\nWrote test predictions to {out}")

with zipfile.ZipFile(submission_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(out, arcname="answer.csv")

print(f"Wrote submission zip to {submission_zip}")

all_metrics, final = compute_all_metrics(valid_df, valid_answer_df)

with open(os.path.join(metrics_dir, "all_metrics.json"), "w") as f:
    json.dump(all_metrics, f, indent=2)

print("\nOfficial style scores " "(weak macro F1) on the public valid set:")

for k, v in all_metrics["official_weak_macro_f1_per_task_dialect"].items():
    print(f"  {k}: {v:.4f}")

print(f"  final score: {final:.4f}")

print("\nAll metrics, confusion matrices, " f"and models saved under:\n  {run_dir}")

Run ID: 20260923-195111_deberta-v3-large_ensemble
Outputs will be written under: C:\Users\X\Desktop\X\ALTA\runs\20260923-195111_deberta-v3-large_ensemble
Using device: cuda
Precision: bf16=True fp16=False
Model: microsoft/deberta-v3-large
Gradient accumulation: 4
Seeds: [42, 1337, 2026]

Seed 42 =

Finetuning for task: sentiment (seed 42)


Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

{'loss': '2.791', 'grad_norm': '12.98', 'learning_rate': '6.243e-08', 'epoch': '0.07231'}
{'loss': '2.799', 'grad_norm': '5.021', 'learning_rate': '1.275e-07', 'epoch': '0.1446'}
{'loss': '2.759', 'grad_norm': '12.9', 'learning_rate': '1.925e-07', 'epoch': '0.2169'}
{'loss': '2.717', 'grad_norm': '14.54', 'learning_rate': '2.575e-07', 'epoch': '0.2892'}
{'loss': '2.307', 'grad_norm': '11.91', 'learning_rate': '3.225e-07', 'epoch': '0.3615'}
{'loss': '1.673', 'grad_norm': '25.61', 'learning_rate': '3.876e-07', 'epoch': '0.4338'}
{'loss': '1.18', 'grad_norm': '33.25', 'learning_rate': '4.526e-07', 'epoch': '0.5061'}
{'loss': '1.339', 'grad_norm': '46.27', 'learning_rate': '5.176e-07', 'epoch': '0.5785'}
{'loss': '1.159', 'grad_norm': '103.1', 'learning_rate': '5.826e-07', 'epoch': '0.6508'}
{'loss': '1.021', 'grad_norm': '18.39', 'learning_rate': '6.477e-07', 'epoch': '0.7231'}
{'loss': '1.231', 'grad_norm': '59.25', 'learning_rate': '7.127e-07', 'epoch': '0.7954'}
{'loss': '0.9134', 'gr

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '1.23', 'grad_norm': '11.43', 'learning_rate': '7.164e-07', 'epoch': '1.012'}
{'loss': '0.9727', 'grad_norm': '7.404', 'learning_rate': '7.152e-07', 'epoch': '1.084'}
{'loss': '1.102', 'grad_norm': '107.1', 'learning_rate': '7.136e-07', 'epoch': '1.156'}
{'loss': '0.9729', 'grad_norm': '130.2', 'learning_rate': '7.117e-07', 'epoch': '1.228'}
{'loss': '0.789', 'grad_norm': '0.2941', 'learning_rate': '7.094e-07', 'epoch': '1.301'}
{'loss': '0.9787', 'grad_norm': '0.4254', 'learning_rate': '7.068e-07', 'epoch': '1.373'}
{'loss': '0.9689', 'grad_norm': '18.24', 'learning_rate': '7.038e-07', 'epoch': '1.445'}
{'loss': '0.9119', 'grad_norm': '81.46', 'learning_rate': '7.005e-07', 'epoch': '1.518'}
{'loss': '0.9195', 'grad_norm': '3.946', 'learning_rate': '6.968e-07', 'epoch': '1.59'}
{'loss': '0.9869', 'grad_norm': '157.2', 'learning_rate': '6.929e-07', 'epoch': '1.662'}
{'loss': '0.9763', 'grad_norm': '30.29', 'learning_rate': '6.885e-07', 'epoch': '1.735'}
{'loss': '0.9751', 'grad

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (91 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '3061', 'train_samples_per_second': '21.68', 'train_steps_per_second': '1.356', 'train_loss': '1.85', 'epoch': '6'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Wrote test predictions to C:\Users\X\Desktop\X\ALTA\runs\20260923-195111_deberta-v3-large_ensemble\answer.csv
Wrote submission zip to C:\Users\X\Desktop\X\ALTA\runs\20260923-195111_deberta-v3-large_ensemble\submission.zip

Official style scores (weak macro F1) on the public valid set:
  sentiment-en-AU: 0.9322
  sentiment-en-UK: 0.9636
  sarcasm-en-AU: 0.8101
  sarcasm-en-UK: 0.7769
  final score: 0.8546

All metrics, confusion matrices, and models saved under:
  C:\Users\X\Desktop\X\ALTA\runs\20260923-195111_deberta-v3-large_ensemble
